In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

In [2]:
df = pd.read_csv('Social_Network_Ads.csv')

In [3]:
df.head()

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0


## Understanding the Dataset


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   User ID          400 non-null    int64 
 1   Gender           400 non-null    object
 2   Age              400 non-null    int64 
 3   EstimatedSalary  400 non-null    int64 
 4   Purchased        400 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 15.8+ KB


In [5]:
print(f"The percentage of 0 class is {df['Purchased'].value_counts()[0]/400*100}")
print(f"The percentage of 1 class is {df['Purchased'].value_counts()[1]/400*100}")

The percentage of 0 class is 64.25
The percentage of 1 class is 35.75


This classification distribution is suitable for implementing logistic regression.

Here's why:

**Balanced enough**: With 64.25% of class 0 and 35.75% of class 1, your dataset is not heavily imbalanced. Logistic regression tends to work well when the classes are reasonably balanced. Although it’s not perfectly balanced, this ratio is quite acceptable for many real-world scenarios.

Class **imbalance considerations**: If your classes were more imbalanced (e.g., 90% vs 10%), you might need to apply techniques like class weighting, oversampling the minority class, or undersampling the majority class to prevent the model from being biased towards the majority class.

In [6]:
# Use the following code to drop the 'User ID' column from the DataFrame.
df = df.drop('User ID', axis=1)

## Method to implement Scaling

In [7]:
# Custom StandardScaler implemented from scratch
class CustomStandardScaler:
    def __init__(self):
        self.mean = None
        self.std = None

    def fit(self, X):
        """Calculate the mean and standard deviation for scaling."""
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)

    def transform(self, X):
        """Apply the standard scaling transformation."""
        return (X - self.mean) / self.std

    def fit_transform(self, X):
        """Fit and transform in one go."""
        self.fit(X)
        return self.transform(X)

In [8]:
# Function to process columns based on dtype
def process_dataframe(df):
    # Creating a copy of the dataframe to store transformed values
    transformed_df = pd.DataFrame()
    for col in df.columns:
        if df[col].dtype == 'object':
            # Creating an object of sklearn.preprocessing._encoders.OneHotEncoder
            ohe = OneHotEncoder(sparse_output=False, drop=None)
            # To transform you need to pass a DataFrame containing a single row or a single column.
            encoded = ohe.fit_transform(df[[col]]) # Encoded is nothing but a list of lists. The dimension of 2nd list depends on the number of categories in the column.
            # The output of encoded columns would be ['Gender_Female', 'Gender_Male']
            # ohe.categories_ output is [array(['Female', 'Male'], dtype=object)]
            encoded_columns = [f"{col}_{cat}" for cat in ohe.categories_[0]]
            # Creating a new dataframe that would be responsible to store these columns.
            encoded_df = pd.DataFrame(encoded, columns=encoded_columns)
            transformed_df = pd.concat([transformed_df, encoded_df], axis=1)
        else:
            # Apply custom standard scaling to numerical columns
            scaler = CustomStandardScaler()
            scaled_values = scaler.fit_transform(df[[col]])
            transformed_df[col] = scaled_values
    return transformed_df

In [9]:
# We need to drop the target column before standarizing the dataset
target_column = 'Purchased'
dataset_beforescaling = df.drop(target_column, axis = 1)
transformed_df = process_dataframe(dataset_beforescaling)
transformed_df.head()

,Gender_Female,Gender_Male,Age,EstimatedSalary
0,0.0,1.0,-1.781797,-1.490046
1,0.0,1.0,-0.253587,-1.460681
2,1.0,0.0,-1.113206,-0.785290
3,1.0,0.0,-1.017692,-0.374182
4,0.0,1.0,-1.781797,0.183751


In [10]:
from sklearn.model_selection import train_test_split
X = transformed_df.values
y = df[target_column].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Creating Logistic Regression from Scratch

In [13]:
import numpy as np

class LogisticRegressionScratch:
    def __init__(self, learning_rate=0.01, n_iters=1000):
        self.learning_rate = learning_rate
        self.n_iters = n_iters
        self.weights = None
        self.bias = None

    def sigmoid(self, z):
        """Sigmoid activation function"""
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        """Fit the logistic regression model using gradient descent"""
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in range(self.n_iters):
            # Linear model
            linear_model = np.dot(X, self.weights) + self.bias
            # Apply sigmoid function
            y_predicted = self.sigmoid(linear_model)

            # Gradient calculations
            # The value of dw won't be a single value, the shape would be like w
            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            # The value of db will be similar to b
            db = (1 / n_samples) * np.sum(y_predicted - y)

            # Update weights and bias
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            if _ % 10 == 0:
              loss = -np.mean(y * np.log(y_predicted) + (1 - y) * np.log(1 - y_predicted))
              print(f"At {_} iteration the loss value is :\t{round(loss,3)}\n")

    def predict(self, X):
        """Predict binary labels (0 or 1) for a given input X"""
        linear_model = np.dot(X, self.weights) + self.bias
        y_predicted = self.sigmoid(linear_model)
        # Convert probabilities to binary labels
        y_predicted_labels = [1 if i > 0.5 else 0 for i in y_predicted]
        return np.array(y_predicted_labels)

    def accuracy(self, y_true, y_pred):
        """Calculate the accuracy of predictions"""
        accuracy = np.sum(y_true == y_pred) / len(y_true)
        return accuracy

    def confusion_matrix(self, y_true, y_pred):
        """Calculate the confusion matrix"""
        TP = np.sum((y_true == 1) & (y_pred == 1))
        TN = np.sum((y_true == 0) & (y_pred == 0))
        FP = np.sum((y_true == 0) & (y_pred == 1))
        FN = np.sum((y_true == 1) & (y_pred == 0))

        return np.array([[TP, FP],[FN, TN]])

In [14]:
# Example usage of the custom Logistic Regression
if __name__ == "__main__":
    model = LogisticRegressionScratch(learning_rate=0.1, n_iters=300)
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)
    #print("Predictions:", predictions)
    print(f"The accuracy of the model is: {model.accuracy(y_test,predictions)}")
    print(f"The confusion matrix is: \n{model.confusion_matrix(y_test,predictions)}")

At 0 iteration the loss value is :	0.693

At 10 iteration the loss value is :	0.589

At 20 iteration the loss value is :	0.528

At 30 iteration the loss value is :	0.49

At 40 iteration the loss value is :	0.465

At 50 iteration the loss value is :	0.447

At 60 iteration the loss value is :	0.434

At 70 iteration the loss value is :	0.424

At 80 iteration the loss value is :	0.416

At 90 iteration the loss value is :	0.41

At 100 iteration the loss value is :	0.405

At 110 iteration the loss value is :	0.4

At 120 iteration the loss value is :	0.397

At 130 iteration the loss value is :	0.394

At 140 iteration the loss value is :	0.391

At 150 iteration the loss value is :	0.389

At 160 iteration the loss value is :	0.387

At 170 iteration the loss value is :	0.386

At 180 iteration the loss value is :	0.384

At 190 iteration the loss value is :	0.383

At 200 iteration the loss value is :	0.382

At 210 iteration the loss value is :	0.381

At 220 iteration the loss value is :	0.38

At 2

Given the confusion matrix for your **Social Network Ads** dataset:

\begin{bmatrix}
21 & 2 \\
7 & 50
\end{bmatrix}

### Interpretation of this Confusion Matrix:

- **True Positives (TP = 21)**:
   - These are individuals who **actually made a purchase** (`Purchased = 1`), and the model **correctly predicted** that they would make a purchase.

- **False Positives (FP = 2)**:
   - These are individuals who **did not make a purchase** (`Purchased = 0`), but the model **incorrectly predicted** that they would make a purchase (false alarm). This is also known as a **Type I error**.

- **False Negatives (FN = 7)**:
   - These are individuals who **actually made a purchase** (`Purchased = 1`), but the model **incorrectly predicted** that they would not make a purchase. This is known as a **Type II error**.

- **True Negatives (TN = 50)**:
   - These are individuals who **did not make a purchase** (`Purchased = 0`), and the model **correctly predicted** that they would not make a purchase.

### Insights:

- **Accuracy (88.75%)**: The model correctly predicted about 89% of the test cases.
- **Precision (91.3%)**: When the model predicted that a customer would make a purchase, it was correct 91.3% of the time.
- **Recall (75%)**: The model identified 75% of actual purchasers, meaning it missed some (7 false negatives).
- **Specificity (96.2%)**: The model was very good at identifying individuals who would not make a purchase.
- **F1 Score (82.3%)**: This score indicates that the model performs well overall, balancing precision and recall.

This analysis suggests that the model is quite accurate and good at predicting purchases, although it could improve in identifying all potential purchasers (boosting recall). The false positives are low (only 2 cases), which is beneficial if predicting non-purchases accurately is important.

Link of the Kaggle Dataset used:
https://www.kaggle.com/datasets/dragonheir/logistic-regression/data